# 09 - Trust-adjusted Rating and Hidden Gems (PuneBite Analyzer)

**Objective:** build a fair way to rank restaurants and to find *hidden gems*: genuinely good places that few people know about yet.
**Input:** `data/processed/restaurants_clean.csv`
**Output:** `ml/scoring.py` (already written), `data/processed/restaurants_scored.csv`, and the API snippet for the dashboard
**Syllabus link:** Unit III (distributions, z-scores, sampling variability, business intelligence KPIs) and Unit I-II (feature engineering).

## The question
A restaurant with a 4.6 rating from 30 reviews and one with 4.4 from 3,000 reviews: which is "better", and which is more of a secret? Raw stars cannot answer either question. This notebook builds two scores:

1. **`trust_rating`**: the rating, lightly smoothed by how much evidence (votes) backs it. Used for sorting lists.
2. **`peer_z`**: how far a restaurant's rating is above restaurants with a *similar number of votes*. Used to find hidden gems.

> **A surprise from the data (section 1):** Zomato's displayed rating is already pulled down for restaurants with few votes. That changes how we should design the gem score, so we start by showing the evidence.

In [ ]:
import os, sys, warnings
sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit

from ml import scoring as sc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/processed/restaurants_clean.csv")
rated = df[df["has_rating"]].copy()
print("Restaurants:", len(df), "| Rated:", len(rated))

## 1. Raw ratings depend on the number of votes

If ratings were just noisy measurements of quality, restaurants with few votes would be *scattered* around the average (some very high, some very low). Let us look.

In [ ]:
fine_bins = [-1, 1, 2, 4, 9, 19, 29, 49, 99, 199, 499, 999, np.inf]
band = pd.cut(rated["votes"], fine_bins)
tab = rated.groupby(band, observed=True).agg(restaurants=("rating", "size"), mean_votes=("votes", "mean"),
                                             mean_rating=("rating", "mean"), std_rating=("rating", "std"),
                                             min_rating=("rating", "min"), max_rating=("rating", "max"))
tab.round(2)

Two things stand out. **(a)** The average rating climbs steadily with votes, from about 3.0 to above 4. **(b)** Restaurants with only a few votes are *tightly bunched* near 3.0 (small standard deviation), not scattered. That is the signature of a rating that has been **pulled towards a default value** when evidence is thin, which is how many platforms compute displayed ratings. (It could also be partly real, since popular places are often better, but the tight clustering is hard to explain without some damping.)

We fit a saturating curve `E[rating | votes] = P + (Q - P) * votes / (votes + k)`. In a "Bayesian average" view, `P` is the default value that thin ratings are pulled towards and `k` is how many votes it takes to overcome it.

In [ ]:
f = lambda v, P, Q, k: P + (Q - P) * v / (v + k)
popt, _ = curve_fit(f, tab["mean_votes"], tab["mean_rating"], p0=[3, 4.2, 20], sigma=1 / np.sqrt(tab["restaurants"]))
P, Q, k = popt
print(f"Fitted default P = {P:.2f}, long-run level Q = {Q:.2f}, half-way point k = {k:.0f} votes")

xs = np.logspace(0, 3.8, 200)
plt.figure(figsize=(9, 5))
plt.scatter(rated["votes"] + 1, rated["rating"], s=6, alpha=0.18, label="restaurants")
plt.scatter(tab["mean_votes"] + 1, tab["mean_rating"], color="red", zorder=3, label="band averages")
plt.plot(xs, f(xs, *popt), color="red", linewidth=2, label="fitted curve")
plt.xscale("log"); plt.xlabel("Votes (log scale)"); plt.ylabel("Displayed rating")
plt.title("Displayed rating vs number of votes"); plt.legend(); plt.show()

**Why this matters for design.** The textbook fix for "a 5.0 from 3 votes shouldn't beat a 4.5 from 3,000" is to shrink ratings towards the average in proportion to missing evidence. Here that has *already been done by the platform*, in a way that pushes low-vote restaurants **down**. If we shrank again, we would punish exactly the restaurants we want to surface. Let us measure that.

## 2. What a second round of shrinkage would do

`trust = v/(v+m) * rating + m/(v+m) * C`, where `C` is the average rating and `m` is the "strength" of the shrinkage in votes. We compare several values of `m` and look at the **top 10%** of restaurants.

In [ ]:
C = rated["rating"].mean()
n_top = int(len(rated) * 0.10)
top_raw = set(rated["rating"].nlargest(n_top).index)

rows = []
for m in [0, 10, 20, 50, 100, 200]:
    t = sc.trust_rating(rated["rating"], rated["votes"], m=m, c=C) if m > 0 else rated["rating"]
    top = set(t.nlargest(n_top).index)
    rows.append({"m": m, "Spearman vs raw": stats.spearmanr(rated["rating"], t)[0],
                 "top-10% overlap with raw": len(top & top_raw) / n_top,
                 "share of top 10% with < 100 votes": (rated.loc[list(top), "votes"] < 100).mean()})
sens = pd.DataFrame(rows).set_index("m")
sens.round(3)

As `m` grows, restaurants with fewer than 100 votes disappear from the top 10%. For a hidden-gem finder that is the wrong direction, so we do **not** use heavy shrinkage for gems. We keep a *light* version (`m = 20`) only as a tie-breaker for sorting lists, and we build the gem score differently (section 4).

## 3. `trust_rating`: light smoothing for sorting (m = 20)

In [ ]:
sc_df = sc.add_scores(df)
r = sc_df[sc_df["has_rating"]].copy()

examples = r[(r["votes"].isin([8, 25, 60, 150, 400, 1500])) & (r["rating"] >= 3.9)].groupby("votes").head(1)
examples[["name", "votes", "rating", "trust_rating"]].sort_values("votes").round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(r["rating"], r["trust_rating"], s=6, alpha=0.3); ax[0].plot([2, 5], [2, 5], "r--")
ax[0].set_xlabel("Displayed rating"); ax[0].set_ylabel("trust_rating (m=20)"); ax[0].set_title("Light smoothing")
ax[1].hist(r["trust_rating"] - r["rating"], bins=40); ax[1].set_title("Change from smoothing"); ax[1].set_xlabel("trust_rating - rating")
plt.tight_layout(); plt.show()
print("Spearman (rating vs trust_rating):", round(stats.spearmanr(r["rating"], r["trust_rating"])[0], 3))

## 4. `peer_z`: compare each restaurant with its peers

Instead of adjusting the rating, we ask: **"is this restaurant better than other restaurants that have about as many votes?"** We split restaurants into fixed vote bands and compute the z-score of each rating inside its band:

`peer_z = (rating - band mean) / band standard deviation`

A z of 1.5 means "1.5 standard deviations above its peers", which is roughly the top 7% of that band. Because peers have the same amount of evidence (and the same platform damping), the comparison is fair.

In [ ]:
band_stats = r.groupby("vote_band").rating.agg(["count", "mean", "std"]).reindex(sc.VOTE_LABELS).dropna()
band_stats.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
sns.boxplot(data=r, x="vote_band", y="rating", order=band_stats.index, ax=ax[0], color="#FF9A3C")
ax[0].set_title("Rating inside each vote band"); ax[0].tick_params(axis="x", rotation=35); ax[0].set_xlabel("")
ax[1].hist(r["peer_z"].dropna(), bins=60); ax[1].axvline(sc.Z_MIN, color="red", linestyle="--", label=f"gem threshold z >= {sc.Z_MIN}")
ax[1].set_title("peer_z distribution"); ax[1].legend()
plt.tight_layout(); plt.show()

### Is `peer_z` just a disguised vote count?
A good score should not be driven by popularity. Among restaurants with a credible number of votes (20 to 300), the raw rating correlates with votes, but `peer_z` should not.

In [ ]:
mid = r[r["votes"].between(sc.MIN_VOTES, sc.MAX_VOTES)]
print(f"Restaurants with {sc.MIN_VOTES}-{sc.MAX_VOTES} votes: {len(mid)}")
print("Spearman(votes, rating):  ", round(stats.spearmanr(mid["votes"], mid["rating"])[0], 3))
print("Spearman(votes, peer_z):  ", round(stats.spearmanr(mid["votes"], mid["peer_z"])[0], 3))

## 5. The Hidden Gem rule

A restaurant is a **hidden gem** when all three hold:

| Condition | Parameter | Reason |
|---|---|---|
| Clearly good | rating >= `MIN_RATING` (3.8) | no "gem" that is just average |
| Better than its peers | `peer_z` >= `Z_MIN` (1.5) | top ~7% among restaurants with similar votes |
| Low visibility but credible | `MIN_VOTES` (20) <= votes <= `MAX_VOTES` (300) | enough evidence to trust, few enough to be a secret |

In [ ]:
gems = r[r["hidden_gem"]]
print("Hidden gems found:", len(gems), f"({len(gems) / len(r):.1%} of rated restaurants)")
print("Median votes:", int(gems["votes"].median()), "| median rating:", gems["rating"].median(),
      "| median price for two: Rs", int(gems["cost_capped"].median()))

### How sensitive is the list to the thresholds?
We vary each threshold and report how many gems we get and how much the list overlaps with the default list (Jaccard similarity: 1.0 = identical).

In [ ]:
default_set = set(gems.index)
rows = []
for z in [1.0, 1.25, 1.5, 2.0]:
    for mx in [150, 300, 500]:
        for mr in [3.6, 3.8, 4.0]:
            s2 = sc.add_scores(df, z_min=z, max_votes=mx, min_rating=mr)
            ids = set(s2[s2["hidden_gem"]].index)
            rows.append({"z_min": z, "max_votes": mx, "min_rating": mr, "gems": len(ids),
                         "overlap_with_default": len(ids & default_set) / len(ids | default_set)})
grid = pd.DataFrame(rows)
display(grid[(grid.min_rating == 3.8)].pivot(index="z_min", columns="max_votes", values="gems"))
display(grid[(grid.min_rating == 3.8)].pivot(index="z_min", columns="max_votes", values="overlap_with_default").round(2))

Gem counts change smoothly rather than jumping: a stricter z or rating gives a shorter, high-confidence list, and a looser one a longer list. We keep the defaults as a balanced middle (about 1.5% of rated restaurants).

## 6. What do hidden gems look like?
We compare three groups: **hidden gems**, **established stars** (rating 4.0+ with more than 300 votes), and **other peers** (20-300 votes, not gems).

In [ ]:
r["group"] = np.where(r["hidden_gem"], "Hidden gem",
              np.where((r["rating"] >= 4.0) & (r["votes"] > sc.MAX_VOTES), "Established star",
              np.where(r["votes"].between(sc.MIN_VOTES, sc.MAX_VOTES), "Other peers", "Other")))
cmp_ = r[r["group"] != "Other"]
prof = cmp_.groupby("group").agg(restaurants=("rating", "size"), median_rating=("rating", "median"),
                                 median_votes=("votes", "median"), median_cost=("cost_capped", "median"),
                                 table_booking=("table_booking_recommended", "mean"), full_bar=("full_bar_available", "mean"),
                                 outdoor=("outdoor_seating", "mean"), home_delivery=("home_delivery", "mean"))
prof.round(2)

In [ ]:
types = cmp_.groupby("group").primary_type.value_counts(normalize=True).rename("share").reset_index()
top_types = cmp_["primary_type"].value_counts().head(6).index
plt.figure(figsize=(9, 4.2))
sns.barplot(data=types[types.primary_type.isin(top_types)], x="primary_type", y="share", hue="group", order=top_types)
plt.title("Restaurant types by group"); plt.xlabel(""); plt.ylabel("Share of group"); plt.xticks(rotation=20); plt.show()

print("Localities with the most hidden gems:")
gems["locality"].value_counts().head(8)

## 7. The gem list

In [ ]:
cols = ["name", "locality", "primary_type", "rating", "votes", "cost_capped", "peer_z", "trust_rating"]
gems.sort_values("gem_score", ascending=False)[cols].head(15).round(2)

In [ ]:
# Top 3 gems in the five localities with the most gems
for loc in gems["locality"].value_counts().head(5).index:
    print("\n" + loc)
    display(gems[gems.locality == loc].sort_values("gem_score", ascending=False)[["name", "primary_type", "rating", "votes", "cost_capped"]].head(3).round(2))

## 8. Checks and limits

- **No ground truth.** We cannot prove these restaurants are "really" better; we can only say they stand out against comparable restaurants. Visiting a few is the real test.
- **Platform damping is inferred.** The curve in section 1 is consistent with the displayed rating being pulled towards about 3 when votes are few, but real quality differences also exist. Our peer comparison works either way, because it never relies on the exact cause.
- **The star-percentage columns do not reconstruct the rating** (their implied average does not match the displayed rating, correlation only about 0.3), so we did not use them.
- **Snapshot in time.** A gem today may have 1,000 votes next year.

## 9. Save the scores

In [ ]:
out_cols = ["url", "name", "locality", "rating", "votes", "trust_rating", "vote_band", "peer_z", "hidden_gem", "gem_score", "gem_rank_locality"]
out = sc_df[out_cols].copy()
out["beats_percent"] = out["peer_z"].apply(lambda z: sc.beats_percent(z) if pd.notna(z) else np.nan)
os.makedirs("../data/processed", exist_ok=True)
out.to_csv("../data/processed/restaurants_scored.csv", index=False)
print("Saved ../data/processed/restaurants_scored.csv", out.shape, "| gems:", int(out["hidden_gem"].sum()))
out[out["hidden_gem"]].sort_values("gem_score", ascending=False).head(3)

## 10. Plug it into the MongoDB loader and the API

**A. In `backend/load_data.py`**, score the cleaned data before inserting (the loader already reads the cleaned CSV):

```python
from ml.scoring import add_scores, beats_percent

df = add_scores(df)                       # adds trust_rating, peer_z, hidden_gem, gem_score, gem_rank_locality
df["beats_percent"] = df["peer_z"].apply(lambda z: beats_percent(z) if pd.notna(z) else None)
```

Make sure NaN becomes `None` before inserting (MongoDB cannot store NaN in some drivers), and add indexes:

```python
restaurants.create_index([("hidden_gem", 1), ("gem_score", -1)])
restaurants.create_index("locality")
```

**B. In `backend/app.py`**, an endpoint shaped for the React dashboard (it expects `name, locality, rating, votes, cost, type, cuisines[], beats, url`):

```python
@app.get("/api/hidden-gems")
def hidden_gems():
    q = {"hidden_gem": True}
    locality = request.args.get("locality")
    if locality:
        q["locality"] = locality
    limit = min(int(request.args.get("limit", 60)), 200)
    docs = restaurants.find(q, {"_id": 0}).sort("gem_score", -1).limit(limit)
    return jsonify([{
        "name": d["name"], "locality": d["locality"], "rating": d["rating"], "votes": d["votes"],
        "cost": d["cost_capped"], "type": d["primary_type"], "cuisines": d["cuisines"][:3],
        "beats": d["beats_percent"], "url": d.get("url", ""),
    } for d in docs])
```

If `cuisines` is stored as a pipe-separated string rather than a list, use `d["cuisines"].split("|")[:3]`.

**C. Default sorting.** Use `trust_rating` (descending) as the default sort in the restaurant explorer and the chatbot, so a 4.7 from 4 votes does not outrank a 4.5 from 800.

## 11. Summary and findings

### Key findings

1. **Displayed ratings are already damped for low-vote restaurants.** Restaurants with 2 to 4 votes average 3.03 and are tightly bunched (standard deviation 0.10), while those with 1,000+ votes average 4.18. A saturating curve fits well: default value about 3.04, long-run level about 4.05, half-way point about 65 votes. This is consistent with a platform that pulls thin ratings towards about 3. (Real quality differences also exist, so we describe it as consistent with damping, not proven.)
2. **A second round of shrinkage would hide the very restaurants we want to find.** Among the top 10% by score, the share with fewer than 100 votes falls from 11.4% (raw rating) to 5.9% (m = 20), 2.7% (m = 50) and 0.7% (m = 100). So `trust_rating` uses a light m = 20 and is only a sorting aid.
3. **`peer_z` removes the popularity effect.** Among restaurants with 20 to 300 votes, the rating correlates with votes at about 0.45 (Spearman), but `peer_z` correlates at about 0.00. The score measures "better than comparable restaurants", not "more reviewed".
4. **114 hidden gems** (1.5% of rated restaurants, about 3% of the 3,746 restaurants with 20 to 300 votes). The typical gem has 71 votes, a 4.2 rating and costs about Rs 400 for two.
5. **Gems are not scaled-down fine dining.** Compared with established stars (rating 4.0+, over 300 votes), gems have the same median rating (4.2) but about a tenth of the votes and less than half the price (Rs 400 vs Rs 1,000). Far fewer take table bookings (16% vs 37%) or serve a full bar (12% vs 44%). They are more often cafes, takeaways, delivery kitchens and dessert parlours (11% to 14% each, versus 6% or less for stars).
6. **Where they are:** Viman Nagar (12), Koregaon Park (9), Hinjawadi (8) and Baner (8) have the most.
7. **The list is stable in the middle.** Tightening to z >= 2 leaves 39 gems, loosening to z >= 1.0 gives 320, and the default (z >= 1.5, up to 300 votes, rating >= 3.8) sits in between.

### Limits
- There is no ground truth, so "gem" means "stands out against comparable restaurants", not "proven better".
- One snapshot in time; today's gem may be famous next year.
- The star-percentage columns do not reconstruct the displayed rating, so they were not used.

### Done in this phase
`ml/scoring.py` (with `beats_percent`), `data/processed/restaurants_scored.csv`, and the loader and API snippets in section 10. The React dashboard already shows these gems.